# ELT Medallion Architecture: CoinGecko Top Coins

This notebook implements a medallion architecture (Bronze → Silver → Gold) for CoinGecko top coins data:

- **Bronze**: Raw data ingested from S3 via Auto Loader (`cloudFiles` format) with schema inference and schema evolution (`addNewColumns`). Uses PySpark Structured Streaming with `trigger(availableNow=True)` to process all available files in batches.
- **Silver**: Deduplicated, current-state-only records using SCD Type 1 via streaming `foreachBatch` + `MERGE INTO`. Only the most recent record per coin (by `load_timestamp`) is kept.
- **Gold**: Business-level aggregations as pre-computed Delta tables for analytics and reporting, including per-coin market metrics and market-wide overview statistics.

## Environment Configuration

The notebook supports **dev** and **prod** environments:
- **Dev**: `crypto-data-project-dev` catalog (for testing and development)
- **Prod**: `crypto-data-project` catalog (for production data)

To switch environments, change the `environment` variable in the **Environment Configuration** cell (next cell).

> **Note**: Native streaming tables and materialized views are not available on this compute. The notebook uses equivalent PySpark Structured Streaming + Delta table patterns that achieve the same medallion architecture.

In [0]:
# Environment Configuration
# Change 'environment' to switch between dev and prod

environment = "dev"  # Options: "dev", "prod"

# Environment-specific catalog configuration
env_config = {
    "dev": {
        "catalog": "crypto-data-project-dev",
        "bronze_schema": "bronze_crypto",
        "silver_schema": "silver_crypto",
        "gold_schema": "gold_crypto",
        "s3_source": "s3://databricks-data-landing-s3/databricks/coingecko/top_coins/"
    },
    "prod": {
        "catalog": "crypto-data-project",
        "bronze_schema": "bronze_crypto",
        "silver_schema": "silver_crypto",
        "gold_schema": "gold_crypto",
        "s3_source": "s3://databricks-data-landing-s3/databricks/coingecko/top_coins/"
    }
}

# Set current environment config
config = env_config[environment]
catalog = config["catalog"]
bronze_schema = config["bronze_schema"]
silver_schema = config["silver_schema"]
gold_schema = config["gold_schema"]
s3_source = config["s3_source"]

print(f"Environment: {environment.upper()}")
print(f"Catalog: {catalog}")
print(f"Schemas: {bronze_schema}, {silver_schema}, {gold_schema}")
print(f"S3 Source: {s3_source}")

In [0]:
# Create schemas for the medallion architecture
# Bronze: raw landing data | Silver: cleaned & deduplicated | Gold: business aggregations

print(f"Using catalog: {catalog}")

for schema in [bronze_schema, silver_schema, gold_schema]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.{schema}")
    print(f"  Schema ready: {catalog}.{schema}")

In [0]:
# Bronze Layer: Auto Loader streaming ingestion from S3 to Delta table
# Auto Loader provides: schema inference, schema evolution (addNewColumns), incremental file discovery
# NOTE: Streaming Tables (CREATE OR REFRESH STREAMING TABLE) are not available on this compute.
#       Using PySpark Structured Streaming with Auto Loader + Delta tables as the equivalent.

import time

bronze_table = f"`{catalog}`.{bronze_schema}.top_coins_raw"

# Create a UC volume for Auto Loader schema and streaming checkpoints
spark.sql(f"CREATE VOLUME IF NOT EXISTS `{catalog}`.{bronze_schema}.streaming_metadata")
checkpoint_loc = f"/Volumes/{catalog}/{bronze_schema}/streaming_metadata/bronze_top_coins/checkpoint"
schema_loc = f"/Volumes/{catalog}/{bronze_schema}/streaming_metadata/bronze_top_coins/schema"

# Start Auto Loader streaming query (continuously ingests new files from S3)
bronze_query = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", schema_loc)
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .load(s3_source)
    .writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_loc)
    .option("mergeSchema", "true")
    .outputMode("append")
    .trigger(availableNow=True)
    .queryName("bronze_top_coins_loader")
    .toTable(bronze_table)
)

# Wait for the batch to complete (availableNow processes all available data then stops)
bronze_query.awaitTermination()

count = spark.sql(f"SELECT COUNT(*) AS cnt FROM `{catalog}`.{bronze_schema}.top_coins_raw").collect()[0]['cnt']
print(f"Bronze Delta table '{bronze_table}' loaded with {count} rows")

In [0]:
# Silver Layer: SCD Type 1 deduplication via streaming MERGE
# SCD1: only the most recent record per coin (by load_timestamp) is kept - no history
# Uses foreachBatch with MERGE INTO for upsert (insert new coins, update existing if newer)
# NOTE: AUTO CDC is not available on this compute; using PySpark streaming + MERGE as equivalent.

import time

bronze_table = f"`{catalog}`.{bronze_schema}.top_coins_raw"
silver_table = f"`{catalog}`.{silver_schema}.top_coins"

# Create checkpoint location for silver streaming query
spark.sql(f"CREATE VOLUME IF NOT EXISTS `{catalog}`.{silver_schema}.streaming_metadata")
silver_checkpoint = f"/Volumes/{catalog}/{silver_schema}/streaming_metadata/scd1_checkpoint"

# Clean slate: drop existing table and checkpoint from any failed attempts
spark.sql(f"DROP TABLE IF EXISTS {silver_table}")
dbutils.fs.rm(silver_checkpoint, True)

# Create silver table with same schema as bronze (empty target for MERGE)
spark.sql(f"CREATE TABLE IF NOT EXISTS {silver_table} LIKE {bronze_table}")

# SCD1 upsert function: MERGE new batches, update existing coins only if newer
def scd1_upsert(batch_df, batch_id):
    import uuid
    if batch_df.limit(1).count() == 0:
        return
    # Use unique view name per batch to avoid stale DataFrame references on Spark Connect
    view_name = f"`_silver_batch_{uuid.uuid4()}`"
    batch_df.createOrReplaceTempView(view_name)
    # Deduplicate within batch: keep only latest record per id, then MERGE (SCD1)
    spark.sql(f"""
        MERGE INTO {silver_table} AS t
        USING (
            SELECT * EXCEPT (_rn) FROM (
                SELECT *, ROW_NUMBER() OVER (PARTITION BY id ORDER BY load_timestamp DESC) AS _rn
                FROM {view_name}
            ) WHERE _rn = 1
        ) AS s
        ON t.id = s.id
        WHEN MATCHED AND s.load_timestamp > t.load_timestamp THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

# Start streaming MERGE from bronze to silver
silver_query = (spark.readStream
    .table(bronze_table)
    .writeStream
    .foreachBatch(scd1_upsert)
    .option("checkpointLocation", silver_checkpoint)
    .trigger(availableNow=True)
    .queryName("silver_scd1_upsert")
    .start()
)

# Wait for SCD1 batch to complete (availableNow processes all data then stops)
silver_query.awaitTermination()

count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {silver_table}").collect()[0]['cnt']
distinct_ids = spark.sql(f"SELECT COUNT(DISTINCT id) AS cnt FROM {bronze_table}").collect()[0]['cnt']
print(f"Silver table '{silver_table}' - {count} unique coins (bronze has {distinct_ids} distinct ids)")

In [0]:
# Gold Layer: Per-coin business metrics with market context
# Pre-computed Delta table with aggregations (materialized views not available on this compute)

spark.sql(f"""
CREATE OR REPLACE TABLE `{catalog}`.{gold_schema}.coin_market_summary
COMMENT 'Gold layer: per-coin enriched business metrics with market-level context'
AS
SELECT
  id,
  name,
  symbol,
  current_price,
  market_cap,
  market_cap_rank,
  total_volume,
  ROUND(price_change_percentage_24h, 2) AS price_change_pct_24h,
  price_change_24h,
  high_24h,
  low_24h,
  circulating_supply,
  total_supply,
  max_supply,
  ath,
  ath_change_percentage AS ath_change_pct,
  atl,
  atl_change_percentage AS atl_change_pct,
  -- Market dominance: this coin's market cap as % of total tracked market cap
  ROUND(market_cap * 100.0 / NULLIF(SUM(market_cap) OVER (), 0), 2) AS market_dominance_pct,
  -- Volume to market cap ratio (liquidity indicator)
  ROUND(total_volume * 100.0 / NULLIF(market_cap, 0), 2) AS volume_to_mcap_ratio_pct,
  -- Intraday price range as % of low
  ROUND((high_24h - low_24h) / NULLIF(low_24h, 0) * 100, 2) AS intraday_range_pct,
  -- Circulating supply as % of total supply
  ROUND(circulating_supply * 100.0 / NULLIF(total_supply, 0), 2) AS supply_circulated_pct,
  -- Price trend classification
  CASE
    WHEN price_change_percentage_24h > 5 THEN 'Strong Gainer'
    WHEN price_change_percentage_24h > 0 THEN 'Gainer'
    WHEN price_change_percentage_24h > -5 THEN 'Loser'
    ELSE 'Strong Loser'
  END AS price_trend_category,
  load_timestamp AS last_ingested_at
FROM `{catalog}`.{silver_schema}.top_coins
""")

print(f"Created gold table: {catalog}.{gold_schema}.coin_market_summary")

In [0]:
# Gold Layer: Market-level aggregate statistics
# Single-row summary of the entire tracked market

spark.sql(f"""
CREATE OR REPLACE TABLE `{catalog}`.{gold_schema}.market_overview
COMMENT 'Gold layer: market-level aggregate statistics for top coins'
AS
SELECT
  COUNT(*) AS total_coins_tracked,
  SUM(market_cap) AS total_market_cap,
  SUM(total_volume) AS total_24h_volume,
  ROUND(AVG(price_change_percentage_24h), 2) AS avg_price_change_pct_24h,
  MAX(price_change_percentage_24h) AS top_gainer_pct_24h,
  MIN(price_change_percentage_24h) AS top_loser_pct_24h,
  ROUND(SUM(CASE WHEN price_change_percentage_24h > 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_gainers,
  ROUND(SUM(CASE WHEN price_change_percentage_24h < 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_losers,
  MAX(load_timestamp) AS last_refreshed_at
FROM `{catalog}`.{silver_schema}.top_coins
""")

print(f"Created gold table: {catalog}.{gold_schema}.market_overview")

In [0]:
# Verify row counts and sample data from each layer
print(f"Verifying {environment.upper()} environment...\n")

print("=== Bronze Layer (raw) ===")
bronze_count = spark.sql(f"SELECT COUNT(*) AS cnt FROM `{catalog}`.{bronze_schema}.top_coins_raw").collect()[0]['cnt']
print(f"Bronze row count: {bronze_count}")
display(spark.sql(f"SELECT id, name, current_price, market_cap_rank, load_timestamp FROM `{catalog}`.{bronze_schema}.top_coins_raw ORDER BY load_timestamp DESC LIMIT 5"))

print("\n=== Silver Layer (SCD1 - deduplicated) ===")
silver_count = spark.sql(f"SELECT COUNT(*) AS cnt FROM `{catalog}`.{silver_schema}.top_coins").collect()[0]['cnt']
print(f"Silver row count (unique coins): {silver_count}")
display(spark.sql(f"SELECT id, name, current_price, market_cap_rank, load_timestamp FROM `{catalog}`.{silver_schema}.top_coins ORDER BY market_cap_rank LIMIT 5"))

print("\n=== Gold Layer - Coin Market Summary ===")
gold_count = spark.sql(f"SELECT COUNT(*) AS cnt FROM `{catalog}`.{gold_schema}.coin_market_summary").collect()[0]['cnt']
print(f"Gold coin_market_summary rows: {gold_count}")
display(spark.sql(f"SELECT id, name, symbol, current_price, market_cap_rank, market_dominance_pct, price_change_pct_24h, price_trend_category FROM `{catalog}`.{gold_schema}.coin_market_summary ORDER BY market_cap_rank LIMIT 10"))

print("\n=== Gold Layer - Market Overview ===")
display(spark.sql(f"SELECT * FROM `{catalog}`.{gold_schema}.market_overview"))